In [ ]:
import os
import zipfile
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. THIẾT LẬP ĐƯỜNG DẪN & ĐỌC DỮ LIỆU GỐC
DATA_ZIP_PATH = '../data/dataset.zip'
DATA_CSV_PATH = '../data/dataset.csv'
MODELS_DIR = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs('../data/', exist_ok=True)

# Giải nén dataset nếu chưa có sẵn file CSV
if not os.path.exists(DATA_CSV_PATH) and os.path.exists(DATA_ZIP_PATH):
    with zipfile.ZipFile(DATA_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('../data/')
    print("-> Đã giải nén dataset.zip")

df = pd.read_csv(DATA_CSV_PATH)
print(f"Kích thước dữ liệu gốc: {df.shape}")

# 2. TÁCH ĐẶC TRƯNG & CHIA TRAIN/TEST (CHỐNG RÒ RỈ DỮ LIỆU)
# Loại bỏ cột định danh loan_id, tách nhãn loan_status
X = df.drop(columns=['loan_id', 'loan_status'])
y = df['loan_status'].map({'y': 1, 'n': 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Tập Train: {X_train.shape}, Tập Test: {X_test.shape}")

# 3. XÂY DỰNG PIPELINE TIỀN XỬ LÝ (FIT CHỈ TRÊN TẬP TRAIN)
num_cols = ['applicantincome', 'coapplicantincome', 'loanamount', 'loan_amount_term', 'credit_history']
cat_cols = ['gender', 'married', 'dependents', 'education', 'self_employed', 'property_area']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit trên Train và transform cho cả Train và Test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print(f"Kích thước X_train_processed: {X_train_processed.shape}")
print(f"Kích thước X_test_processed: {X_test_processed.shape}")

# 4. XUẤT SCHEMA.JSON CHO BE VALIDATE VÀ FE DỰNG FORM
schema = {
    "features": {
        "gender": {"type": "string", "allowed_values": ["male", "female"]},
        "married": {"type": "string", "allowed_values": ["yes", "no"]},
        "dependents": {"type": "string", "allowed_values": ["0", "1", "2", "3+"]},
        "education": {"type": "string", "allowed_values": ["graduate", "not graduate"]},
        "self_employed": {"type": "string", "allowed_values": ["yes", "no"]},
        "applicantincome": {"type": "float", "min": 0, "max": 100000},
        "coapplicantincome": {"type": "float", "min": 0, "max": 50000},
        "loanamount": {"type": "float", "min": 0, "max": 1000},
        "loan_amount_term": {"type": "float", "allowed_values": [12.0, 36.0, 60.0, 84.0, 120.0, 180.0, 240.0, 300.0, 360.0, 480.0]},
        "credit_history": {"type": "float", "allowed_values": [0.0, 1.0]},
        "property_area": {"type": "string", "allowed_values": ["urban", "semiurban", "rural"]}
    },
    "target": {
        "name": "loan_status",
        "labels": {"0": "Rejected", "1": "Approved"}
    }
}

schema_path = os.path.join(MODELS_DIR, 'schema.json')
with open(schema_path, 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=4, ensure_ascii=False)

# 5. LƯU CÁC TẬP DỮ LIỆU VÀ PREPROCESSOR
joblib.dump(preprocessor, os.path.join(MODELS_DIR, 'preprocessor.joblib'))
joblib.dump((X_train, X_test, y_train, y_test, X_train_processed, X_test_processed), 
            os.path.join(MODELS_DIR, 'processed_splits.joblib'))

print("-> Hoàn thành 02_preprocess.ipynb! Đã lưu schema.json và processed_splits.joblib thành công.")

Thông tin tổng quan về dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   loan_id            614 non-null    object 
 1   gender             601 non-null    object 
 2   married            611 non-null    object 
 3   dependents         599 non-null    object 
 4   education          614 non-null    object 
 5   self_employed      582 non-null    object 
 6   applicantincome    614 non-null    int64  
 7   coapplicantincome  614 non-null    float64
 8   loanamount         592 non-null    float64
 9   loan_amount_term   600 non-null    float64
 10  credit_history     564 non-null    float64
 11  property_area      614 non-null    object 
 12  loan_status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


None


5 dòng đầu tiên của dataset:


,loan_id,gender,married,dependents,education,self_employed,applicantincome,coapplicantincome,loanamount,loan_amount_term,credit_history,property_area,loan_status
0,lp001002,male,no,0,graduate,no,5849,0.0,NaN,360.0,1.0,urban,y
1,lp001003,male,yes,1,graduate,no,4583,1508.0,128.0,360.0,1.0,rural,n
2,lp001005,male,yes,0,graduate,yes,3000,0.0,66.0,360.0,1.0,urban,y
3,lp001006,male,yes,0,not graduate,no,2583,2358.0,120.0,360.0,1.0,urban,y
4,lp001008,male,no,0,graduate,no,6000,0.0,141.0,360.0,1.0,urban,y
